In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, root_mean_squared_log_error


In [3]:
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [4]:
#log

import logging

logging.basicConfig(
    filename='app.log',
    filemode='a',
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', 
    datefmt='%Y-%m-%d %H:%M:%S',  
)

logger = logging.getLogger()



## Load Data

In [6]:
house = pd.read_csv("house_price_prediction_dataset.csv").drop(["Id"], axis=1)
house.head(5)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Basic Analysis

- Understand the size of the data
- Basic info of the dataset
- Check if there is any missing features in the dataset
- capture unique values of categorical and numerical columns and write to files
- describe the numerical features in the dataset

In [7]:
object_columns = house.select_dtypes('object').columns.to_list()
numerical_columns = house.select_dtypes('number').columns.to_list()

In [ ]:
from  scipy import stats 
house[numerical_columns].var().sort_values().round(2)
x=pd.Series([1,2,3])
x.var()


np.float64(1.0)

In [8]:
def check_missing_values(df):
    missing_feature_counts = df.isna().sum()
    missing_feature_df = (missing_feature_counts[missing_feature_counts >0].to_frame()).reset_index()

    #print(f"total number of missing features: {missing_feature_df.shape[0]}")
    missing_feature_df.columns = ["Feature", "Count"]

    missing_feature_df["null_percentage"] = round((missing_feature_df["Count"]/df.shape[0]) *100, 2)
    missing_feature_df["Type"] = missing_feature_df["Feature"].map(df.dtypes)
    
    return missing_feature_df.sort_values(by="null_percentage", ascending= False)

In [9]:
def get_dataset_details(df):
    
    object_columns = df.select_dtypes('object').columns.to_list()
    numerical_columns = df.select_dtypes('number').columns.to_list()
    
    missing_features = check_missing_values(df)

    with open ("object_columns.txt", 'w') as file:
        for obj_col in object_columns:
            #file.writelines(f"{obj_col}:\n")
            file.write(f"{obj_col} Feature contains {house[obj_col].nunique()} unique values:\n \n")
            file.write(f"{house[obj_col].value_counts().to_string()} \n \n")
    
    with open ("numerical_columns.txt", 'w') as file:
        for num_col in numerical_columns:
            #file.writelines(f"{obj_col}:\n")
            file.write(f"{num_col} Feature contains {house[num_col].nunique()} unique values:\n \n")
            if house[num_col].nunique() <= 10:
                file.write(f"{house[num_col].value_counts().to_string()} \n \n")

    return {"object_columns": object_columns,
            "numerical_columns": numerical_columns,
            "missing_features": missing_features}


In [10]:
def highly_correlated_features(data, threshold):
    corr_matrix = data.corr().abs().round(2)
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k =1).astype(bool))
    high_corr_features = [ (col, row) for col in upper_triangle.columns for row in upper_triangle.index if upper_triangle[col][row] > threshold]
    return high_corr_features

## EDA

In [11]:
def eda(data):
    sample, features = data.shape
    #print(f"The house price dataset contains {sample} samples and {features} features")
    data_deatils = get_dataset_details(data)

    object_columns = data_deatils["object_columns"]
    numerical_columns = data_deatils["numerical_columns"]
    missing_features = data_deatils["missing_features"]

    skewness = data[numerical_columns].skew().sort_values(ascending=False) # using third moment about the mean
    highly_skewed_columns = skewness[(skewness > 1) | (skewness < -1)].index.to_list()
    #mod_skewed_columns = skewness[(skewness > 1) or (skewness <-1) ].index.to_list()

    kurtosis = data[numerical_columns].kurtosis().sort_values(ascending=False)
    leptokurtic = kurtosis[kurtosis > 0].index.to_list() # heavy tail, higher probability of extreme values

    #correlation
    threshold = 0.7
    high_correlated_features = highly_correlated_features(data[numerical_columns], threshold)


    logger.info(f"The house price dataset contains {sample} samples and {features} features\n")
    logger.info(f"Total Number of object colums: {len(object_columns)}\n")
    logger.info(f"Total Number of Numerical colums: {len(numerical_columns)}\n")
    logger.info(f"Total Number of Missing Features:\n {len(missing_features)}\n")
    logger.info(f"Missing Features:\n {missing_features}\n")
    logger.info("===" * 30)

    logger.info("\n Numerical column summary:\n")
    logger.info(f"{(house.describe().T).to_string()}\n")

    logger.info("===" * 30)
    logger.info("\n Skewness:\n")
    logger.info(skewness.to_string())

    logger.info(f"\n\n No of highly Skewed Features: {len(highly_skewed_columns)}\n")
    logger.info(f"\n Highly Skewed Features: {highly_skewed_columns}\n")

    logger.info("===" * 30)
    logger.info("\n excess Kustosis:\n")
    logger.info(f"count= {len(leptokurtic)},\nFeatures= {leptokurtic}\n")

    logger.info("===" * 30)
    logger.info(f"\n Highly correlated features based on threshold {threshold}:\n {high_correlated_features}")
    
    
    print(f"Total Number of object colums: {len(object_columns)}")
    print(f"Total Number of Numerical colums: {len(numerical_columns)}")
    print(f"Total Number of Missing Features: {len(missing_features)}\n")
    #print(f"Missing Features:\n {missing_features}\n")
    print(f"\n No of highly Skewed Features: {len(highly_skewed_columns)}\n")
    print(f"\n Highly Skewed Features: {highly_skewed_columns}\n")

    print(f"Heavy Kustosis: count= {len(leptokurtic)},\nFeatures= {leptokurtic}")

    print(f"Highly correlated features based on threshold {threshold}: {high_correlated_features}")
    
    # print("===" * 50)

    # print("Numerical column summary")
    # print(house.describe().T)



In [12]:
eda(house)

Total Number of object colums: 43
Total Number of Numerical colums: 37
Total Number of Missing Features: 19


 No of highly Skewed Features: 20


 Highly Skewed Features: ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'SalePrice', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea']

Heavy Kustosis: count= 27,
Features= ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'LotFrontage', 'BsmtHalfBath', 'TotalBsmtSF', 'BsmtFinSF1', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'SalePrice', '1stFlrSF', 'GrLivArea', 'WoodDeckSF', 'BedroomAbvGr', 'MSSubClass', 'OverallCond', 'GarageArea', 'TotRmsAbvGrd', 'BsmtUnfSF', 'GarageCars', 'OverallQual']
Highly correlated features based on threshold 0.7: [('1stFlrSF', 'TotalBsmtSF'), ('TotRmsAbvGrd', 'GrLivArea'), ('GarageYrBlt'

# Univariate Analysis

- Analyze the distribution of individual features
- Outlier Identification
- Missing value treatment

In [1335]:
# def hist_plot(data):
#     no_features = data.shape[1]
#     no_cols = 4
#     no_rows = (no_features // 4) +1

#     fig, axes = plt.subplots(no_rows, no_cols, figsize=(24,no_rows * 6))
#     axes = axes.flatten()

#     for i, col in enumerate(data.columns.to_list()):
#         feature_mean = data[col].mean().round(2)
#         feature_median = data[col].median()
#         #print(feature_mean, feature_median)

#         axes[i].hist(data[col], bins = 20, edgecolor='k')
#         axes[i].axvline(feature_mean, color = 'red', linestyle="--", label=f"Mean:{feature_mean}")
#         axes[i].axvline(feature_median, color='green', linestyle = "-", label = f"Median:{feature_median}")

#         axes[i].set_title(f"Distribution of '{col}'")
#         axes[i].set_xlabel(col)
#         axes[i].set_ylabel("Frequency")
    
#     for j in range(no_features, len(axes)):
#         axes[j].set_visible(False)


In [1336]:
# def box_plot(data):
#     no_features = data.shape[1]
#     no_cols = 4
#     no_rows = (no_features // 4) +1

#     fig, axes = plt.subplots(no_rows, no_cols, figsize=(24,no_rows * 6))
#     axes = axes.flatten()

#     for i, col in enumerate(data.columns.to_list()):
#         sns.boxplot(data[col], color='lightblue', ax=axes[i])
#         #axes[i].hist(data[col], bins = 20, edgecolor='k')
#         axes[i].set_title(f"Boxplot of '{col}'")
#         # axes[i].set_xlabel(col)
#         # axes[i].set_ylabel("Frequency")
    
#     for j in range(no_features, len(axes)):
#         axes[j].set_visible(False)


In [13]:
def Numerical_column_distribution(data):
    no_features = data.shape[1]

    fig, axes = plt.subplots(no_features, 2, figsize=(14, no_features * 6))
    
    #axes = axes.flatten()

    for i, col in enumerate(data.columns.to_list()):

        feature_mean = data[col].mean().round(2)
        feature_median = data[col].median()
        #print(feature_mean, feature_median)

        sns.histplot(data[col], bins = 20, edgecolor='k', kde=True, ax=axes[i][0])
        axes[i][0].axvline(feature_mean, color = 'red', linestyle="--", label=f"Mean:{feature_mean}")
        axes[i][0].axvline(feature_median, color='green', linestyle = "-", label = f"Median:{feature_median}")

        axes[i][0].set_title(f"Distribution of '{col}'")
        axes[i][0].set_xlabel(col)
        axes[i][0].set_ylabel("Frequency")


        sns.boxplot(data[col], color='lightblue', ax=axes[i][1])
        #axes[i].hist(data[col], bins = 20, edgecolor='k')
        axes[i][1].set_title(f"Boxplot of '{col}'")
        axes[i][1].set_xlabel(col)
        # axes[i].set_ylabel("Frequency")
    
    for j in range(no_features, len(axes)):
        axes[j].set_visible(False)

## Numerical Column Distribution

In [1338]:


#Numerical_column_distribution(house[numerical_columns])

## Categorical Column Distribution

In [1339]:
def categorical_column_distribution(data):
    no_features = data.shape[1]
    no_cols = 2
    no_rows = no_features//no_cols +1 
    
    print(no_features)
    fig, axes = plt.subplots(no_rows, no_cols, figsize = (16,60))
    axes = axes.flatten()

    for i, col in enumerate(data.columns.to_list()):
        value_counts = data[col].value_counts()
        #print(value_counts)
        axes[i].bar(value_counts.index, value_counts.values)

    for j in range(no_features, len(axes)):
        axes[j].set_visible(False)
    

In [1340]:
#categorical_column_distribution(house[object_columns])

# Feature Engineering

The data contains 1460 samples and 79 features, No of features are quite high. 
We will use Filter methods to determine, 
- low variance fetures from each independent features
- compare variance between groups vs. variance within group - find F-statistics
- MI (Mutual information) - dependency between feature X and target Y

question - if I add new feature derived from existing should it be done after preprocessing. that means after splitting, morever we nwould need scaling of those features as well, so should not it be done before importance and preprocessing

# Data Cleanup

In [14]:
 #### get columns which has null value count more that 50%
missing_feature_counts = check_missing_values(house)
cols_to_drop = missing_feature_counts[missing_feature_counts["null_percentage"] > 40]["Feature"].to_list()
logger.info(f"Features which contain more than 40% null value: {cols_to_drop}")
house.drop(cols_to_drop, axis=1, inplace=True)

#hightly corrlated features - ('1stFlrSF', 'TotalBsmtSF'), ('TotRmsAbvGrd', 'GrLivArea'), ('GarageYrBlt', 'YearBuilt'), ('GarageArea', 'GarageCars')

#house.drop(["1stFlrSF", "TotRmsAbvGrd", "GarageYrBlt", "GarageCars"], axis=1, inplace= True)

In [15]:
def data_cleanup(df):
    logger.info(f"The house price dataset contains {df.shape[0]} samples and {df.shape[1]} features")
    print(f"drop columns which contains more than 40% null values in the training set: {cols_to_drop}")
    #df.drop(cols_to_drop, axis=1, inplace=True)

    sample, features = df.shape
    logger.info(f"\nAfter deleting the feature columns which contain more than 40% null values, the dataset contains {sample} samples and {features} features")

    missing_feature_counts = check_missing_values(df)
    logger.info(f"\nmissing features after deleting the features which contains more than 40% missing values:\n {missing_feature_counts}")

    logger.info(f"\nNumber of Missing values: {len(check_missing_values(df))}")
    logger.info(f"\nAfter clenup: The house price dataset contains {df.shape[0]} samples and {df.shape[1]} features")

    # encode 'CentralAir' as boolean, 

    df['CentralAir'] = df['CentralAir'].map({'Y': 1, 'N': 0})

   
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['HouseRemodAge'] = df['YrSold']  - df['YearRemodAdd']

    df["TotalSF"] = (df["TotalBsmtSF"] +df["1stFlrSF"] + df["2ndFlrSF"])

    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
    

    return df

In [17]:
cleaned_df = data_cleanup(house)
cleaned_df.head()

drop columns which contains more than 40% null values in the training set: ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HouseAge,HouseRemodAge,TotalSF,TotalBath
0,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,1,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,0,2,2008,WD,Normal,208500,5,5,2566,3.5
1,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,1,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,0,5,2007,WD,Normal,181500,31,31,2524,2.5
2,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,1,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,0,9,2008,WD,Normal,223500,7,6,2706,3.5
3,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,1,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,0,2,2006,WD,Abnorml,140000,91,36,2473,2.0
4,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,1,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,0,12,2008,WD,Normal,250000,8,8,3343,3.5


In [18]:
cleaned_df.shape

(1460, 78)

In [1345]:
# As per random forest model below features does not have any importance, So drop them from the feature list
#"Electrical","Heating_Wall", "Functional", "RoofMatl", "RoofStyle", "SaleType_ConLw", "GarageType_Basment", "SaleCondition_Family", 
#"HouseStyle", "Condition2", "Neighborhood", "LotConfig_Inside", "Utilities", "Street","MSZoning", "YrSold","MiscVal", 
#"3SsnPorch", "EnclosedPorch","KitchenAbvGr", "BsmtHalfBath", "BsmtFullBath", "LowQualFinSF", "BldgType", "Condition1", "PavedDrive"


# cleaned_df.drop(["Electrical","Heating", "Functional", "RoofMatl", "RoofStyle", "SaleType", "GarageType", 
#                  "SaleCondition", "HouseStyle", "Condition2", "Neighborhood", "LotConfig", "Utilities", "Street",
#                  "MSZoning","MiscVal", "3SsnPorch", "EnclosedPorch","KitchenAbvGr", "BsmtHalfBath", "BsmtFullBath", 
#                  "LowQualFinSF", "BldgType", "Condition1", "PavedDrive", "MoSold"], inplace=True, axis=1)



In [1346]:
cleaned_df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HouseAge,HouseRemodAge,TotalSF,TotalBath
0,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,1,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,0,2,2008,WD,Normal,208500,5,5,2566,3.5
1,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,1,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,0,5,2007,WD,Normal,181500,31,31,2524,2.5
2,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,1,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,0,9,2008,WD,Normal,223500,7,6,2706,3.5
3,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,1,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,0,2,2006,WD,Abnorml,140000,91,36,2473,2.0
4,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,1,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,0,12,2008,WD,Normal,250000,8,8,3343,3.5


# Preprocessing +Feature Selection PCA

In [19]:
from sklearn.impute import SimpleImputer


X = cleaned_df.drop(["SalePrice"], axis=1) 
y = cleaned_df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logger.info(f"Total features: {X_train.shape[1]}")
cat_cols = X_train.select_dtypes(include="object").columns.to_list()
num_cols = X_train.select_dtypes(exclude="object").columns.to_list()
#num_cols = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
            # 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'CentralAir', '2ndFlrSF', 
            # 'GrLivArea', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'Fireplaces', 'GarageArea', 'WoodDeckSF', 
            # 'OpenPorchSF', 'ScreenPorch', 'PoolArea', "YrSold", "PropertyAge", "YrSinceRemod"]

logger.info(f"Categorical_Columns: {cat_cols}")
logger.info(f"Numerical Columns: {num_cols}")

logger.info("==" * 30)

logger.info("Update column list after randomforest feature importance")
# 'CentralAir' - already encoded as boolean, 

cat_onehotencode = ['MSZoning','Street', 'Utilities','LotConfig', 'Neighborhood', 'Condition1','Condition2', 'BldgType', 'HouseStyle',\
                    'RoofStyle', 'RoofMatl', 'Heating', 'Electrical' , 'GarageType', 'SaleType', 'SaleCondition' ]  #review Utilities as it contains all same value

# cat_ordinal = ['LotShape', 'LandContour', 'LandSlope', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',\
#                'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual' ,'Functional', 'GarageFinish', \
#                 'GarageQual', 'GarageCond', 'PavedDrive']

#cat_onehotencode = ['YrSold']
cat_ordinal = ['LotShape', 'LandContour', 'LandSlope', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',\
               'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual' ,'GarageFinish', \
                'GarageQual', 'GarageCond',]



numeric_transformer = Pipeline(
    steps=[
        #('mean_imputer', SimpleImputer(strategy='mean')),
        ('median_imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
        ]
)

categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
])



# # include PCA

pca = PCA(n_components=0.95)
pipeline_pca = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('PCA', pca)
    ]
)

# logger.info("==" * 30)
# logger.info("preprocess the data using PCA")
# X_train_scaled_pca = pipeline_pca.fit_transform(X_train)
# X_test_scaled_pca = pipeline_pca.transform(X_test)

# logger.info(f"After transformation shape of X_train scaled: {X_train_scaled_pca.shape}")
# logger.info(f"After transformation shape of X_test scaled: {X_test_scaled_pca.shape}")

# logger.info(f"Explained variance ratio:\n {pca.explained_variance_ratio_}")
# logger.info(f"Total Explained variance ratio: {sum(pca.explained_variance_ratio_)}")



# ## without pca
# logger.info("==" * 30)
# logger.info("preprocess the data without PCA")
# X_train_scaled = preprocessor.fit_transform(X_train)
# X_test_scaled = preprocessor.transform(X_test)

# logger.info(f"After transformation shape of X_train scaled: {X_train_scaled.shape}")
# logger.info(f"After transformation shape of X_test scaled: {X_test_scaled.shape}")

# feature_names =[name.split("__")[1] for name in preprocessor.get_feature_names_out()]
# logger.info(f"Preprocesed FeatureNames: {feature_names}")




In [20]:
pipeline_pca.get_feature_names_out

<bound method Pipeline.get_feature_names_out of Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('median_imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                     

# Baseline model

## Linear Regression

In [ ]:
from sklearn.linear_model import Ridge, Lasso

alpha = 0.1


linear_regression = Pipeline(
    steps=[
        ("pre", preprocessor),
        ("lr", LinearRegression(tol=0.0001, fit_intercept=True))
    ]
)

linear_regression_pca = Pipeline(
    steps=[
        ('pre', preprocessor),
        ('PCA', pca),
        ("lr_pca", LinearRegression(tol=0.0001, fit_intercept=True))
    ]
)

linear_regression_ridge = Pipeline(
    steps=[
        ('pre', preprocessor),
        ("lr_ridge", Ridge(tol=0.0001, alpha=alpha, fit_intercept=True))
    ]
)

linear_regression_ridge_pca = Pipeline(
    steps=[
        ('pre', preprocessor),
        ('PCA', pca),
        ("lr_ridge_pca", Ridge(tol=0.0001, alpha=alpha, fit_intercept=True))
    ]
)

linear_regression_lasso = Pipeline(
    steps=[
        ('pre', preprocessor),
        ("lr_lasso", Lasso(tol=0.0001, alpha=alpha, fit_intercept=True))
    ]
)

linear_regression_lasso_pca = Pipeline(
    steps=[
        ('pre', preprocessor),
        ('PCA', pca),
        ("lr_lasso_pca", Lasso(tol=0.0001, alpha=alpha, fit_intercept=True))
    ]
)


model_pipelines = [("linear_regression", linear_regression), 
                   ("linear_regression_pca", linear_regression_pca),
                   ("linear_regression_ridge", linear_regression_ridge),
                   ("linear_regression_ridge_pca", linear_regression_ridge_pca),
                   ("linear_regression_lasso", linear_regression_lasso),
                   ("linear_regression_lasso_pca", linear_regression_lasso_pca)] 

try:
    r2_scores = []

    for name, pipe in model_pipelines:
        pipe.fit(X_train, y_train)
        y_test_pred = pipe.predict(X_test)
        y_train_pred = pipe.predict(X_train)


        r2_score_test = r2_score(y_test,y_test_pred)
        r2_score_train = r2_score(y_train, y_train_pred)

        rmse_test = root_mean_squared_error(y_test,y_test_pred)
        rmse_train = root_mean_squared_error(y_train, y_train_pred)  

        r2_scores.append({
                    "Model": name, 
                    "R2_score-Test": r2_score_test, 
                    "R2_score-Train": r2_score_train,
                    "RMSE-Test": rmse_test, 
                    "RMSE-Train": rmse_train
                    })  

except ValueError as err:
    print(err)

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML

,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train
0,linear_regression,0.713555,0.914318,46873.487329,22606.572403
1,linear_regression_pca,0.839862,0.831683,35047.317539,31684.896329
2,linear_regression_ridge,0.855486,0.905558,33293.632213,23734.057252
3,linear_regression_ridge_pca,0.839863,0.831683,35047.186617,31684.896448
4,linear_regression_lasso,0.713959,0.914317,46840.482167,22606.622789
5,linear_regression_lasso_pca,0.839862,0.831683,35047.276001,31684.896338


## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor_decision_tree = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
])

decision_tree_regrssor = Pipeline(
    steps=[
        ("pre", preprocessor_decision_tree),
        ("decision_tree", DecisionTreeRegressor(min_samples_split=20,random_state=42))
    ]
)

model_pipelines = [("Decision Tree", decision_tree_regrssor)]

try:
    for name, pipe in model_pipelines:
        pipe.fit(X_train, y_train)
        y_test_pred = pipe.predict(X_test)
        y_train_pred = pipe.predict(X_train)


        r2_score_test = r2_score(y_test,y_test_pred)
        r2_score_train = r2_score(y_train, y_train_pred)

        rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
        rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

        r2_scores.append({
                    "Model": name, 
                    "R2_score-Test": r2_score_test, 
                    "R2_score-Train": r2_score_train,
                    "RMSLE-Test": rmse_test, 
                    "RMSLE-Train": rmse_train
                    })  

except ValueError as err:
    print(err)

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.713555,0.914318,46873.487329,22606.572403,NaN,NaN
1,linear_regression_pca,0.839862,0.831683,35047.317539,31684.896329,NaN,NaN
2,linear_regression_ridge,0.855486,0.905558,33293.632213,23734.057252,NaN,NaN
3,linear_regression_ridge_pca,0.839863,0.831683,35047.186617,31684.896448,NaN,NaN
4,linear_regression_lasso,0.713959,0.914317,46840.482167,22606.622789,NaN,NaN
5,linear_regression_lasso_pca,0.839862,0.831683,35047.276001,31684.896338,NaN,NaN
6,Decision Tree,0.793209,0.935256,39826.511808,19651.237052,NaN,NaN
7,Decision Tree,0.793209,0.935256,0.188763,0.101109,NaN,NaN
8,Decision Tree,0.793209,0.935256,NaN,NaN,0.188763,0.101109
9,Decision Tree,0.793209,0.935256,NaN,NaN,0.188763,0.101109


In [1355]:

feature_names_dt =[name.split("__")[1] for name in preprocessor_decision_tree.get_feature_names_out()]
logger.info(f"Decision Tree Preprocesed FeatureNames: {feature_names_dt}")

feature_importance_dt = model_decision_tree.feature_importances_.round(3)
feature_importance_df = pd.DataFrame({
                            "Feature": feature_names_dt, "importance": feature_importance_dt
                            }).sort_values(by='importance', ascending= False)

high_imp_features = feature_importance_df[feature_importance_df['importance'] >0]
#model_decision_tree_pca.feature_importances_.round(3)

logger.info("Feature Importance as per Decision Tree")
logger.info(high_imp_features.to_string())

# Advance Regression

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
])

random_forest_regrssor = Pipeline(
    steps=[
        ("pre", preprocessor),
        ("random_forest", RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_split=20, random_state=42))
    ]
)

model_pipelines = [("Random Forest ", decision_tree_regrssor)]

try:
    for name, pipe in model_pipelines:
        pipe.fit(X_train, y_train)
        y_test_pred = pipe.predict(X_test)
        y_train_pred = pipe.predict(X_train)


        r2_score_test = r2_score(y_test,y_test_pred)
        r2_score_train = r2_score(y_train, y_train_pred)

        rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
        rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

        r2_scores.append({
                    "Model": name, 
                    "R2_score-Test": r2_score_test, 
                    "R2_score-Train": r2_score_train,
                    "RMSLE-Test": rmse_test, 
                    "RMSLE-Train": rmse_train
                    })  

except ValueError as err:
    print(err)

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.713555,0.914318,46873.487329,22606.572403,NaN,NaN
1,linear_regression_pca,0.839862,0.831683,35047.317539,31684.896329,NaN,NaN
2,linear_regression_ridge,0.855486,0.905558,33293.632213,23734.057252,NaN,NaN
3,linear_regression_ridge_pca,0.839863,0.831683,35047.186617,31684.896448,NaN,NaN
4,linear_regression_lasso,0.713959,0.914317,46840.482167,22606.622789,NaN,NaN
5,linear_regression_lasso_pca,0.839862,0.831683,35047.276001,31684.896338,NaN,NaN
6,Decision Tree,0.793209,0.935256,39826.511808,19651.237052,NaN,NaN
7,Decision Tree,0.793209,0.935256,0.188763,0.101109,NaN,NaN
8,Decision Tree,0.793209,0.935256,NaN,NaN,0.188763,0.101109
9,Decision Tree,0.793209,0.935256,NaN,NaN,0.188763,0.101109


In [ ]:
# from sklearn.ensemble import RandomForestRegressor


# # preprocessor_decision_tree = ColumnTransformer(transformers=[
# #     ('cat_onehotencode', OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False), cat_onehotencode),
# #     ('cat_ordinal', OrdinalEncoder(), cat_ordinal)
# # ])

# # ## fit for decision tree
# # X_train_dt = preprocessor_decision_tree.fit_transform(X_train)
# # X_test_dt = preprocessor_decision_tree.transform(X_test)

# params = (
#     {
#         "n_estimators":500, 
#         "min_samples_split":10,
#         "random_state":42
#     }
# )
# model_random_forest = RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_split=20, random_state=42)
# model_random_forest.fit(X_train_dt, y_train)

# r2_scores.append({
#                 "Model": "Random Forest Regression", 
#                 "R2_score-Test": r2_score(y_test, model_random_forest.predict(X_test_dt)), 
#                 "R2_score-Train": r2_score(y_train, model_random_forest.predict(X_train_dt)),
#                 "RMSE-Test": root_mean_squared_log_error(y_test, model_random_forest.predict(X_test_dt)), 
#                 "RMSE-Train": root_mean_squared_log_error(y_train, model_random_forest.predict(X_train_dt))
#                 })

# # with pca features
# model_random_forest_pca = RandomForestRegressor(n_estimators=500, min_samples_split=10,random_state=42)
# model_random_forest_pca.fit(X_train_scaled_pca, y_train)


# r2_scores.append({
#                 "Model": "Random Forest Regression - PCA", 
#                 "R2_score-Test": r2_score(y_test, model_random_forest_pca.predict(X_test_scaled_pca)), 
#                 "R2_score-Train": r2_score(y_train, model_random_forest_pca.predict(X_train_scaled_pca)),
#                 "RMSE-Test": root_mean_squared_log_error(y_test, model_random_forest_pca.predict(X_test_scaled_pca)), 
#                 "RMSE-Train": root_mean_squared_log_error(y_train, model_random_forest_pca.predict(X_train_scaled_pca))
#                 })


# r2_scores_df = pd.DataFrame(r2_scores)
# logger.info(r2_scores_df.to_string())

# root_mean_squared_error(y_test, model_random_forest_pca.predict(X_test_scaled_pca))

37368.28285731614

In [1357]:
r2_scores_df

,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train
0,Linear Regression,0.713555,0.914318,46873.487329,22606.572403
1,Linear Regression - PCA,0.839862,0.831683,35047.317539,31684.896329
2,Ridge Regression,0.857560,0.860045,0.164087,0.139533
3,Ridge Regression - PCA,0.840159,0.831419,0.175350,0.150797
4,Lasso Regression,0.877753,0.882390,0.159977,0.134021
5,Lasso Regression - PCA,0.840034,0.831660,0.177676,0.152465
6,DecisionTree Regression,0.793209,0.935256,0.188763,0.101109
7,Random Forest Regression,0.871119,0.935936,0.159231,0.104227
8,Random Forest Regression - PCA,0.817949,0.957438,0.178647,0.077466


In [1358]:

feature_names_dt =[name.split("__")[1] for name in preprocessor_decision_tree.get_feature_names_out()]
logger.info(f"Random Forest Tree Preprocesed FeatureNames: {feature_names_dt}")

feature_importance_rf = model_random_forest.feature_importances_.round(3)
feature_importance_df = pd.DataFrame({
                            "Feature": feature_names_dt, "importance_dt": feature_importance_dt, 
                            "Feature": feature_names_dt, "importance_rf": feature_importance_rf
                            }).sort_values(by='importance_rf', ascending= False)

high_imp_features = feature_importance_df[(feature_importance_df['importance_dt'] >0) | (feature_importance_df['importance_rf'] >0)]
#model_decision_tree_pca.feature_importances_.round(3)

logger.info("Feature Importance as per Random Forest")
logger.info(feature_importance_df.to_string())

### XGBoost

In [47]:
from xgboost import XGBRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
])

xgboost_regressor = Pipeline(
    steps=[
        ("pre", preprocessor),
        ("random_forest", XGBRegressor(
                eval_metric='rmse', 
                max_depth = 6, 
                n_estimators=500, 
                min_samples_split=10, 
                learning_rate=0.05, 
                random_state=42
                ))
    ]
)

model_pipelines = [("XGBoost Regressor", xgboost_regressor)]

try:
    for name, pipe in model_pipelines:
        pipe.fit(X_train, y_train)
        y_test_pred = pipe.predict(X_test)
        y_train_pred = pipe.predict(X_train)


        r2_score_test = r2_score(y_test,y_test_pred)
        r2_score_train = r2_score(y_train, y_train_pred)

        rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
        rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

        r2_scores.append({
                    "Model": name, 
                    "R2_score-Test": r2_score_test, 
                    "R2_score-Train": r2_score_train,
                    "RMSLE-Test": rmse_test, 
                    "RMSLE-Train": rmse_train
                    })  

except ValueError as err:
    print(err)

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [12:46:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.713555,0.914318,46873.487329,22606.572403,NaN,NaN
1,linear_regression_pca,0.839862,0.831683,35047.317539,31684.896329,NaN,NaN
2,linear_regression_ridge,0.855486,0.905558,33293.632213,23734.057252,NaN,NaN
3,linear_regression_ridge_pca,0.839863,0.831683,35047.186617,31684.896448,NaN,NaN
4,linear_regression_lasso,0.713959,0.914317,46840.482167,22606.622789,NaN,NaN
5,linear_regression_lasso_pca,0.839862,0.831683,35047.276001,31684.896338,NaN,NaN
6,XGBoost Regressor,0.920351,0.999410,NaN,NaN,0.13209,0.012982


In [1359]:
from xgboost import XGBRegressor


numeric_transformer = Pipeline(
    steps=[
        #('mean_imputer', SimpleImputer(strategy='mean'))
        ('median_imputer', SimpleImputer(strategy='median'))
        ]
)

categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
])

## fit for decision tree
X_train_xgboost = preprocessor.fit_transform(X_train)
X_test_xgboost = preprocessor.transform(X_test)

model_xgboost = XGBRegressor(
                eval_metric='rmse', 
                max_depth = 6, 
                n_estimators=500, 
                min_samples_split=10, 
                learning_rate=0.05, 
                random_state=42
                )

model_xgboost.fit(X_train_xgboost, y_train)

r2_scores.append({
                "Model": "XGBoost Regression", 
                "R2_score-Test": r2_score(y_test, model_xgboost.predict(X_test_xgboost)), 
                "R2_score-Train": r2_score(y_train, model_xgboost.predict(X_train_xgboost)),
                "RMSE-Test": root_mean_squared_log_error(y_test, model_xgboost.predict(X_test_xgboost)), 
                "RMSE-Train": root_mean_squared_log_error(y_train, model_xgboost.predict(X_train_xgboost))
                })

# with pca features
# model_decision_tree_pca = DecisionTreeRegressor(min_samples_split=30,random_state=42)
# model_decision_tree_pca.fit(X_train_scaled_pca, y_train)


# r2_scores.append({
#                 "Model": "DecisionTree Regression - PCA", 
#                 "R2_score-Test": r2_score(y_test, model_decision_tree_pca.predict(X_test_scaled_pca)), 
#                 "R2_score-Train": r2_score(y_train, model_decision_tree_pca.predict(X_train_scaled_pca))
#                 })


r2_scores_df = pd.DataFrame(r2_scores)
logger.info(r2_scores_df.to_string())

r2_scores_df

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [11:33:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train
0,Linear Regression,0.713555,0.914318,46873.487329,22606.572403
1,Linear Regression - PCA,0.839862,0.831683,35047.317539,31684.896329
2,Ridge Regression,0.857560,0.860045,0.164087,0.139533
3,Ridge Regression - PCA,0.840159,0.831419,0.175350,0.150797
4,Lasso Regression,0.877753,0.882390,0.159977,0.134021
5,Lasso Regression - PCA,0.840034,0.831660,0.177676,0.152465
6,DecisionTree Regression,0.793209,0.935256,0.188763,0.101109
7,Random Forest Regression,0.871119,0.935936,0.159231,0.104227
8,Random Forest Regression - PCA,0.817949,0.957438,0.178647,0.077466
9,XGBoost Regression,0.920351,0.999410,0.132090,0.012982


In [1360]:
root_mean_squared_error(y_test, model_xgboost.predict(X_test_xgboost))
from sklearn.metrics import root_mean_squared_log_error

root_mean_squared_log_error(y_test, model_xgboost.predict(X_test_xgboost))

0.13208967447280884

### Hyperparameter tuning

In [1361]:
randint(300, 1000)

In [1366]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_params ={
        "n_estimators": randint(300, 1000),
        "max_depth": randint(3, 10),
        "learning_rate": uniform(0.01, 0.2), 
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0.5, 2)
    }

random_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    param_distributions=xgb_params,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_xgboost, y_train)
print(f"Best param: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_}")

best_xgboost = random_search.best_estimator_

print(f"Best rmse: {root_mean_squared_log_error(y_test, best_xgboost.predict(X_test_xgboost))}")


Best param: {'colsample_bytree': np.float64(0.8721230154351118), 'learning_rate': np.float64(0.1000998503939086), 'max_depth': 4, 'n_estimators': 687, 'reg_alpha': np.float64(0.9422017556848528), 'reg_lambda': np.float64(1.6265764356910786), 'subsample': np.float64(0.7541666010159664)}
Best CV accuracy: 0.8717592477798461
Best rmse: 0.13413368165493011


### LightGBM

In [ ]:
# import lightgbm as lgb


# numeric_transformer = Pipeline(
#     steps=[
#         ('mean_imputer', SimpleImputer(strategy='mean'))
#         ]
# )

# categorical_onehot_transformer = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
#         ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
#     ]
# )

# categorical_ordinal_transformer = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
#         ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
#     ]
# )

# preprocessor = ColumnTransformer(transformers=[
#     ('num', numeric_transformer, num_cols),
#     ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
#     ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal)
# ])

# ## fit for decision tree
# X_train_lgb = preprocessor.fit_transform(X_train)
# X_test_lgb = preprocessor.transform(X_test)

# model_lgb = lgb.LGBMRegressor(
#                 eval_metric='rmse', 
#                 max_depth = 6, 
#                 n_estimators=500, 
#                 min_samples_split=10, 
#                 learning_rate=0.05, 
#                 random_state=42
#                 )

# model_lgb.fit(X_train_xgboost, y_train, eval_set=[(X_test_lgb, y_test)])

# r2_scores.append({
#                 "Model": "XGBoost Regression", 
#                 "R2_score-Test": r2_score(y_test, model_lgb.predict(X_test_lgb)), 
#                 "R2_score-Train": r2_score(y_train, model_lgb.predict(X_train_lgb)),
#                 "RMSE-Test": root_mean_squared_log_error(y_test, model_lgb.predict(X_test_lgb)), 
#                 "RMSE-Train": root_mean_squared_log_error(y_train, model_lgb.predict(X_train_lgb))
#                 })

# # with pca features
# # model_decision_tree_pca = DecisionTreeRegressor(min_samples_split=30,random_state=42)
# # model_decision_tree_pca.fit(X_train_scaled_pca, y_train)


# # r2_scores.append({
# #                 "Model": "DecisionTree Regression - PCA", 
# #                 "R2_score-Test": r2_score(y_test, model_decision_tree_pca.predict(X_test_scaled_pca)), 
# #                 "R2_score-Train": r2_score(y_train, model_decision_tree_pca.predict(X_train_scaled_pca))
# #                 })


# r2_scores_df = pd.DataFrame(r2_scores)
# logger.info(r2_scores_df.to_string())

# r2_scores_df

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3532
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 105
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Info] Start training from score 181441.541952
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with

,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train
0,Linear Regression,0.713631,0.914282,46867.302935,22611.324073
1,Linear Regression - PCA,0.839987,0.831675,35033.575875,31685.718237
2,Ridge Regression,0.857567,0.860035,0.164111,0.139551
3,Ridge Regression - PCA,0.840296,0.831410,0.175332,0.150810
4,Lasso Regression,0.877686,0.882372,0.160012,0.134040
5,Lasso Regression - PCA,0.840168,0.831651,0.177652,0.152477
6,DecisionTree Regression,0.793209,0.935256,0.188763,0.101109
7,Random Forest Regression,0.870678,0.935928,0.159219,0.104208
8,Random Forest Regression - PCA,0.815780,0.957575,0.178636,0.077263
9,XGBoost Regression,0.918186,0.999345,0.133519,0.013499


## Saving and loading model using joblib

In [ ]:
test_df = pd.read_csv('test.csv')
test_df.shape

(1459, 80)

In [ ]:
test_df_id = test_df.drop(["Id"], axis=1)

In [ ]:
test_df_id

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0,TA,TA,PConc,TA,TA,No,GLQ,602.0,Unf,0.0,324.0,926.0,GasA,Ex,Y,SBrkr,926,678,0,1604,0.0,0.0,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,1998.0,Fin,2.0,470.0,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,PConc,Gd,TA,No,ALQ,263.0,Unf,0.0,1017.0,1280.0,GasA,Ex,Y,SBrkr,1280,0,0,1280,0.0,0.0,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,1992.0,RFn,2.0,506.0,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,MeadowV,Norm,Norm,Twnhs,2Story,4,7,1970,1970,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Unf,0.0,Unf,0.0,546.0,546.0,GasA,Gd,Y,SBrkr,546,546,0,1092,0.0,0.0,1,1,3,1,TA,5,Typ,0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,MeadowV,Norm,Norm,TwnhsE,2Story,4,5,1970,1970,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,252.0,Unf,0.0,294.0,546.0,GasA,TA,Y,SBrkr,546,546,0,1092,0.0,0.0,1,1,3,1,TA,6,Typ,0,NaN,CarPort,1970.0,Unf,1.0,286.0,TA,TA,Y,0,24,0,0,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1Story,5,7,1960,1996,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,ALQ,1224.0,Unf,0.0,0.0,1224.0,GasA,Ex,Y,SBrkr,1224,0,0,1224,1.0,0.0,1,0,4,1,TA,7,Typ,1,TA,Detchd,1960.0,Unf,2.0,576.0,TA,TA,Y,474,0,0,0,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,SFoyer,5,5,1992,1992,Gable,CompShg,HdBoard,Wd Shng,NaN,0.0,TA,TA,PConc,Gd,TA,Av,GLQ,337.0

In [ ]:
test_df_cleaned = data_cleanup(test_df_id)

drop columns which contains more than 40% null values in the training set: ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']


In [ ]:
test_df_processed = preprocessor.transform(test_df_cleaned)
test_df_processed.shape

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 2, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


(1459, 158)

In [ ]:
import joblib

joblib.dump(best_xgboost, 'best_model.joblib')

['best_model.joblib']

In [ ]:
loaded_model = joblib.load('best_model.joblib')
test_pred = loaded_model.predict(test_df_processed)
test_pred

array([126780.64 , 166331.34 , 185058.48 , ..., 170593.66 , 127332.555,
       226336.23 ], shape=(1459,), dtype=float32)

In [ ]:
test_pred_df = pd.DataFrame({"Id": test_df["Id"], "SalePrice": test_pred})

In [ ]:
test_pred_df

,Id,SalePrice
0,1461,126780.640625
1,1462,166331.343750
2,1463,185058.484375
3,1464,183340.687500
4,1465,184874.625000
...,...,...
1454,2915,85245.531250
1455,2916,77026.656250
1456,2917,170593.656250
1457,2918,127332.554688


In [ ]:
test_pred_df.to_csv("submission.csv", index=False)

Find number od days since last sell, Once it is calculated drop "MoSold", "YrSold", "DateSold" Columns

In [ ]:
# house["DateSold"] = house.apply(lambda row: f"{row['YrSold']}-{row['MoSold']}-01", axis=1)
# house["DateSold"] = pd.to_datetime(house["DateSold"])
# house.head()

In [ ]:
# house["DaysSinceLastSell"] = (pd.Timestamp.today() - house["DateSold"]).dt.days
# house.head()

In [ ]:
# house.drop(["MoSold", "YrSold", "DateSold"], axis=1, inplace=True)
# house.head()

In [ ]:
# house["Age"] = pd.Timestamp.today().year - house["YearBuilt"]

In [ ]:
# house.drop(["YearBuilt"],axis=1, inplace=True)
